In [328]:
import os
from pathlib import Path
import cv2
import numpy as np
import tensorflow as tf
import random
import time
from tensorflow.keras import layers, Model, optimizers, applications
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.utils import register_keras_serializable
from collections import defaultdict

In [329]:
TRAIN_ROOT = '/kaggle/input/project-data/Project Data/Food/Train'
VAL_ROOT   = '/kaggle/input/project-data/Project Data/Food/Validation'

def load_dataset(root_dir):
    image_paths, labels = [], []
    categories = sorted(
        [d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    for label, cat in enumerate(categories):
        cat_path = os.path.join(root_dir, cat)
        for img in Path(cat_path).rglob("*.jpg"):
            image_paths.append(str(img))
            labels.append(label)

    return image_paths, labels, categories

print("Loading dataset...")
train_paths, train_labels, train_categories = load_dataset(TRAIN_ROOT)
val_paths, val_labels, val_categories = load_dataset(VAL_ROOT)

print(f"✓ Train images: {len(train_paths)}")
print(f"✓ Val images: {len(val_paths)}")
print(f"✓ Train classes: {len(train_categories)}")
print(f"✓ Val classes: {len(val_categories)}")

Loading dataset...
✓ Train images: 2538
✓ Val images: 261
✓ Train classes: 85
✓ Val classes: 8


In [330]:
def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = preprocess_input(img.astype(np.float32))
    return img

In [331]:
def pair_generator_contrastive(paths, labels, batch_size=32):
    labels = np.array(labels)

    while True:
        idxs = np.random.choice(len(paths), batch_size * 2, replace=False)
        x1, x2, y = [], [], []

        for i in range(0, len(idxs), 2):
            img1 = preprocess_image(paths[idxs[i]])
            img2 = preprocess_image(paths[idxs[i+1]])

            x1.append(img1)
            x2.append(img2)

            same = labels[idxs[i]] == labels[idxs[i+1]]
            y.append(1.0 if same else 0.0)

        yield (np.array(x1), np.array(x2)), np.array(y)

In [332]:
class L2Norm(layers.Layer):
    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=1)

class DistanceLayer(layers.Layer):
    def call(self, inputs):
        emb1, emb2 = inputs
        return tf.sqrt(tf.reduce_sum(tf.square(emb1 - emb2), axis=1) + 1e-7)

In [333]:
def create_embedding_net():
    base = applications.ResNet50(include_top=False, weights='imagenet',input_shape=(224, 224, 3))

    for layer in base.layers[:-27]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128)(x)       
    embedding = L2Norm()(x)

    return Model(base.input, embedding)

In [334]:
def create_siamese_model():
    embedding_net = create_embedding_net()

    input_a = layers.Input((224, 224, 3))
    input_b = layers.Input((224, 224, 3))

    emb_a = embedding_net(input_a)
    emb_b = embedding_net(input_b)

    distance = DistanceLayer()([emb_a, emb_b])
    model = Model([input_a, input_b], distance)

    return model, embedding_net

In [335]:
def contrastive_loss(y_true, y_pred, margin=1.0):
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean(y_true * tf.square(y_pred) + (1 - y_true) * tf.square(tf.maximum(margin - y_pred, 0)))

In [336]:
siamese_model, embedding_net = create_siamese_model()

siamese_model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss=contrastive_loss
)

siamese_model.summary()

Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_25      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_26      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_18       │ (None, 128)       │ 24,710,656 │ input_layer_25[0… │
│ (Functional)        │                   │            │ input_layer_26[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ distance_layer_6    │ (None)            │          0 │ functional_18[0]… │
│ (DistanceLayer)     │                   │            │ functional_18[1]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,710,656 (94.26 MB)

 Trainable params: 13,208,192 (50.39 MB)

 Non-trainable params: 11,502,464 (43.88 MB)

In [ ]:
train_gen = pair_generator_contrastive(train_paths, train_labels, 32)

start_train = time.time()  

history = siamese_model.fit(
    train_gen,
    steps_per_epoch=len(train_paths) // 32,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=5,
            restore_best_weights=True
        )
    ]
)
end_train = time.time()  

training_time = end_train - start_train
print(f"Training time: ({training_time/60:.2f} minutes)")

Epoch 1/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 50s 292ms/step - loss: 0.0180
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 23s 287ms/step - loss: 0.0152
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 283ms/step - loss: 0.0166
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 283ms/step - loss: 0.0132
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 301ms/step - loss: 0.0196
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 23s 289ms/step - loss: 0.0135
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 278ms/step - loss: 0.0106
Epoch 8/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 284ms/step - loss: 0.0131
Epoch 9/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 284ms/step - loss: 0.0132
Epoch 10/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 285ms/step - loss: 0.0114
Epoch 11/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 22s 282ms/step - loss: 0.0129
Epoch 12/20
26/79 ━━━━━━━━━━━━━━━━━━━━ 14s 277ms/step - loss: 0.0110

In [ ]:
def evaluate_verification_accuracy(embedding_net, paths, labels, num_pairs=1000):
    correct = 0
    labels = np.array(labels)

    for _ in range(num_pairs):
        i1, i2 = np.random.randint(0, len(paths), 2)
        img1 = np.expand_dims(preprocess_image(paths[i1]), axis=0)
        img2 = np.expand_dims(preprocess_image(paths[i2]), axis=0)

        emb1 = embedding_net.predict(img1, verbose=0)
        emb2 = embedding_net.predict(img2, verbose=0)

        dist = np.linalg.norm(emb1 - emb2)
        true_same = labels[i1] == labels[i2]
        pred_same = dist < 0.7

        if true_same == pred_same:
            correct += 1

    return correct / num_pairs

train_verif_acc = evaluate_verification_accuracy(embedding_net, train_paths, train_labels)

print(f"Train verification accuracy: {train_verif_acc*100:.2f}%")

In [ ]:
def group_by_class(paths, labels):
    class_dict = defaultdict(list)
    for p, l in zip(paths, labels):
        class_dict[l].append(p)
    return class_dict

val_class_dict = group_by_class(val_paths, val_labels)

In [ ]:
def get_embedding(embedding_net, image_path):
    img = np.expand_dims(preprocess_image(image_path), axis=0)
    emb = embedding_net.predict(img, verbose=0)
    return emb

def compute_all_embeddings(embedding_net, paths):
    embs = {}
    for i, p in enumerate(paths):
        embs[p] = get_embedding(embedding_net, p)
    return embs

val_embeddings = compute_all_embeddings(embedding_net, val_paths)

In [ ]:
siamese_model.save("/kaggle/working/siamese_model.keras")
embedding_net.save_weights("/kaggle/working/embedding.weights.h5")

In [ ]:
def run_episode_cached(class_dict, embeddings, n_way=5, k_shot=1, threshold=0.9):
    selected_classes = random.sample(list(class_dict.keys()), n_way)
    support_embeddings = {}
    query_class = None
    query_image = None

    for cls in selected_classes:
        images = random.sample(class_dict[cls], k_shot + 1)
        support_imgs = images[:k_shot]
        query_img = images[k_shot]
        embs = [embeddings[img] for img in support_imgs]
        support_embeddings[cls] = np.mean(embs, axis=0)
        query_class = cls
        query_image = query_img

    query_emb = embeddings[query_image]

    min_dist = float("inf")
    pred_class = None
    for cls, ref_emb in support_embeddings.items():
        dist = np.linalg.norm(query_emb - ref_emb)
        if dist < min_dist:
            min_dist = dist
            pred_class = cls

    if min_dist > threshold:
        pred_class = None  
    return pred_class == query_class

In [ ]:
def episodic_evaluation_cached(class_dict, embeddings, n_way=5, k_shot=5, episodes=500):
    correct = 0
    for ep in range(1, episodes + 1):
        if run_episode_cached(class_dict, embeddings, n_way, k_shot):
            correct += 1
    return correct / episodes

start_eval = time.time()
acc_5way_5shot = episodic_evaluation_cached(val_class_dict, val_embeddings, n_way=5, k_shot=5, episodes=500)
end_eval = time.time()

print(f"5-way 5-shot accuracy: {acc_5way_5shot*100:.2f}%")

evaluation_time = end_eval - start_eval
print(f"5-way 5-shot evaluation time: {evaluation_time:.2f} seconds\n")

In [ ]:
acc_5way_10shot = episodic_evaluation_cached(val_class_dict, val_embeddings, n_way=5, k_shot=10, episodes=500)
print(f"5-way 10-shot accuracy: {acc_5way_10shot*100:.2f}%")

In [ ]:
TEST_FOLDER = "/kaggle/input/testcase2/Siamese Case II Test"
EMBEDDING_WEIGHTS = "/kaggle/working/embedding.weights.h5"


def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = preprocess_input(img.astype(np.float32))
    return np.expand_dims(img, axis=0)

@register_keras_serializable()
class L2Norm(layers.Layer):
    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=1)
        
def create_embedding_net():
    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    for layer in base.layers[:-27]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128)(x)
    embedding = L2Norm()(x)
    return Model(base.input, embedding)

embedding_net = create_embedding_net()
embedding_net.load_weights(EMBEDDING_WEIGHTS)


all_images = [os.path.join(TEST_FOLDER, f) for f in os.listdir(TEST_FOLDER) if f.lower().endswith(".jpg")]

anchor_path = None
reference_paths = []

for path in all_images:
    if "anchor" in os.path.basename(path).lower():
        anchor_path = path
    else:
        reference_paths.append(path)

assert anchor_path is not None, "Anchor image not found in the folder!"

anchor_emb = embedding_net.predict(preprocess_image(anchor_path), verbose=0)

reference_embeddings = {}
for ref_path in reference_paths:
    reference_embeddings[ref_path] = embedding_net.predict(preprocess_image(ref_path), verbose=0)


distances = []

for ref_path, ref_emb in reference_embeddings.items():
    dist = np.linalg.norm(anchor_emb - ref_emb)
    distances.append((os.path.basename(ref_path), dist))

distances.sort(key=lambda x: x[1])

threshold = 0.9 
closest_name, min_dist = distances[0]

print(f"Anchor image: {os.path.basename(anchor_path)}\n")
print("Distances to reference images:")
for name, dist in distances:
    print(f"{name}: {dist:.4f}")

if min_dist > threshold:
    print("\nNo match found for the anchor image.")
else:
    print(f"\nMost similar image: {closest_name} (Distance: {min_dist:.4f})")